# 🧠 Truth-Preserving Resume Rewriter — v2 (Improved)

## What changed from v1 and why

| Weakness in v1 | Fix in v2 |
|---|---|
| Fact locking was prompt-only — LLM could ignore it | **Hard schema lock**: fact bank is frozen as a `frozenset` of atomic fact strings before rewriting. Rewrite output is checked against this set programmatically, not just by the LLM. |
| Validator flagged 4 false positives (Education, Summary) due to JSON vs resume formatting mismatch | **Field-level validation**: each output section is checked against the corresponding fact bank field, not against the raw JSON string |
| Validation was detection-only — output was returned even if hallucinations were found | **Reject-and-retry loop**: if validation fails, the output is regenerated with the hallucinated claims listed as an explicit blocklist in the prompt |
| Semantic similarity scores were low (max 0.45) because skills stored as single keywords | **Sentence-level fact representation**: each fact is expanded into a descriptive sentence before embedding, giving much stronger cosine similarity |
| Summary line was LLM-synthesized — not traceable to any single fact | **Template-based Summary**: assembled deterministically from locked facts, not generated freely |
| No gap reporting between candidate and JD | **Gap analysis**: skills/tools in JD not found in fact bank are explicitly listed as gaps |

Get your FREE Groq API key at: https://console.groq.com/keys

## Step 1: Install & Import

In [ ]:
!pip install groq pdfplumber sentence-transformers --quiet
print('✅ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 34.7 MB/s eta 0:00:00
✅ Dependencies installed


In [ ]:
from groq import Groq
import json
import re
import hashlib
import pdfplumber
from sentence_transformers import SentenceTransformer, util

import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)
MODEL  = "llama-3.3-70b-versatile"
MAX_RETRIES = 3   # how many times to regenerate if validation fails

print(f'✅ Groq client ready | model: {MODEL} | max retries: {MAX_RETRIES}')

✅ Groq client ready | model: llama-3.3-70b-versatile | max retries: 3


## Step 2: Load Resume

In [ ]:
def load_resume(file_path: str) -> str:
    """Load resume from PDF or plain-text file."""
    if file_path.endswith('.pdf'):
        text = ''
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + '\n'
        return text.strip()
    else:
        with open(file_path, 'r') as f:
            return f.read().strip()

resume_text = load_resume('/content/Kartik_resume.pdf')  # change filename as needed
print(f'📄 Resume loaded — {len(resume_text)} characters')
print('─' * 60)
print(resume_text[:600])

📄 Resume loaded — 1397 characters
────────────────────────────────────────────────────────────
Cherukuri Venkata Kartik
Bengaluru — karthikcv72@gmail.com — 09937889095 — LinkedIn
Education
PES University Sept 2023 – May 2027
B.Tech in Computer Science
GPA: 8.55/10
Mother’s Public School 2023
Class XII – 90.6%
Mother’s Public School 2020
Class X – 93.5%
Projects
Dynamic Content Stream with Kafka
• Built real-time adaptive streaming pipeline with dynamic topic creation and multi-threaded producer.
• Tech: Python, Apache Kafka, MySQL, Flask.
FullStack AI Assistant
• Developed MERN-based AI assistant with API integration.
• Tech: MongoDB, ExpressJS, ReactJS, NodeJS.
Real-Time Server Monitor


## Step 3: Extract Fact Bank (Structured JSON)

**Fix from v1:** Schema now includes `achievements` and `certifications`. Each project stores its full description as a single string so the validator can match sentences, not just keywords.

In [ ]:
def call_groq(prompt: str, system_msg: str = 'You are a helpful assistant.') -> str:
    """Shared Groq API caller with temperature=0.1 for maximum consistency."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_msg},
            {'role': 'user',   'content': prompt}
        ],
        temperature=0.1,   # lower than v1 (was 0.2) — maximise fidelity to fact bank
        max_tokens=2048
    )
    return response.choices[0].message.content.strip()


def _strip_fences(text: str) -> str:
    """Remove markdown code fences from LLM output."""
    text = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.MULTILINE)
    text = re.sub(r'\s*```$', '', text.strip(), flags=re.MULTILINE)
    return text.strip()


def extract_fact_bank(resume_text: str) -> dict | None:
    """
    Extract a fully-typed structured fact bank from the resume.

    FIX from v1:
    - Each project now stores a unified 'summary' string (name + description + tech)
      so the validator can match full sentences, not fragments.
    - Added 'achievements' and 'certifications' fields.
    - Schema is strictly defined; missing fields default to empty lists.
    """
    prompt = f"""Extract structured facts EXACTLY as they appear in this resume.

STRICT RULES:
- Do NOT add, infer, or embellish anything
- Copy exact strings from the resume — do not paraphrase
- Output ONLY valid JSON with no explanation or markdown fences
- EXPAND all abbreviations in the skills list to their full forms:
    ML → Machine Learning, DL → Deep Learning, NLP → Natural Language Processing,
    AI → Artificial Intelligence, CV → Computer Vision, LLM → Large Language Model,
    RL → Reinforcement Learning, NNs → Neural Networks
  (expand in skills and tech fields only — keep everything else verbatim)
Return this exact schema (use empty list [] if a section is absent):
{{
  "name": "<full name>",
  "contact": {{
    "email": "", "phone": "", "linkedin": "", "location": ""
  }},
  "skills": ["skill1", "skill2"],
  "education": [
    {{
      "degree": "", "institution": "", "period": "", "score": ""
    }}
  ],
  "experience": [
    {{
      "title": "", "company": "", "period": "", "points": [""]
    }}
  ],
  "projects": [
    {{
      "name": "",
      "description": "<exact description as written>",
      "tech": ["tech1", "tech2"],
      "summary": "<name>: <description> Tech: <comma-separated tech>"
    }}
  ],
  "achievements": ["<exact achievement string>"],
  "certifications": ["<exact cert string>"]
}}

Resume:
{resume_text}
"""

    raw = call_groq(prompt, system_msg='You are a precise JSON extractor. Output only valid JSON.')
    raw = _strip_fences(raw)

    try:
        bank = json.loads(raw)
    except json.JSONDecodeError:
        print('❌ JSON parse failed. Raw output:')
        print(raw)
        return None

    # Guarantee all keys exist even if LLM omitted them
    for key, default in [
        ('skills', []), ('education', []), ('experience', []),
        ('projects', []), ('achievements', []), ('certifications', [])
    ]:
        bank.setdefault(key, default)

    return bank


fact_bank = extract_fact_bank(resume_text)

if fact_bank:
    print('✅ Fact bank extracted successfully')
    print(f"   Name      : {fact_bank.get('name')}")
    print(f"   Skills    : {len(fact_bank.get('skills', []))} items")
    print(f"   Education : {len(fact_bank.get('education', []))} entries")
    print(f"   Projects  : {len(fact_bank.get('projects', []))} entries")
    print(f"   Experience: {len(fact_bank.get('experience', []))} entries")
    print('\n' + '─'*60)
    print(json.dumps(fact_bank, indent=2))

✅ Fact bank extracted successfully
   Name      : Cherukuri Venkata Kartik
   Skills    : 18 items
   Education : 3 entries
   Projects  : 4 entries
   Experience: 0 entries

────────────────────────────────────────────────────────────
{
  "name": "Cherukuri Venkata Kartik",
  "contact": {
    "email": "karthikcv72@gmail.com",
    "phone": "09937889095",
    "linkedin": "LinkedIn",
    "location": "Bengaluru"
  },
  "skills": [
    "C",
    "Python",
    "MySQL",
    "HTML/CSS",
    "Verilog (Basic)",
    "Assembly (Basic)",
    "ReactJS",
    "NodeJS",
    "MongoDB",
    "Neo4j",
    "postgreSQL",
    "DSA",
    "OS",
    "CN",
    "DBMS",
    "Machine Learning",
    "Applied Cryptography",
    "Big Data"
  ],
  "education": [
    {
      "degree": "B.Tech in Computer Science",
      "institution": "PES University",
      "period": "Sept 2023 \u2013 May 2027",
      "score": "GPA: 8.55/10"
    },
    {
      "degree": "Class XII",
      "institution": "Mother\u2019s Public School",
  

## Step 4: Build the Atomic Fact Set (Hard Lock)

**NEW in v2 — this is the core novelty fix.**

Every piece of information in the fact bank is decomposed into **atomic fact strings** stored in a `frozenset`. This set acts as the hard constraint:
- It is immutable — nothing can be added to it after creation
- Every claim in the rewritten resume is checked against it programmatically
- A SHA-256 hash of the fact set is computed to prove it has not been tampered with

This replaces the old 4-gram JSON-string matching which caused false positives.

In [ ]:
import hashlib

# Moved from cell KMRByGrI-FfB
ABBREV_MAP = {
    'ml':  'machine learning',
    'dl':  'deep learning',
    'nlp': 'natural language processing',
    'ai':  'artificial intelligence',
    'cv':  'computer vision',
    'llm': 'large language model',
    'rl':  'reinforcement learning',
    'nn':  'neural network',
    'api': 'application programming interface',
    'sql': 'structured query language',
}

def expand_abbreviations(text: str) -> str:
    """Replace known abbreviations with full forms (whole-word, case-insensitive)."""
    for abbrev, full in ABBREV_MAP.items():
        text = re.sub(rf'\b{abbrev}\b', full, text, flags=re.IGNORECASE)
    return text

def build_atomic_fact_set(bank: dict) -> tuple[frozenset, str]:
    """
    Decompose the fact bank into a frozenset of lowercase atomic strings.

    Returns:
        fact_set : frozenset of normalised atomic fact strings
        hash_hex : SHA-256 hash of the sorted fact set (tamper-proof seal)

    FIX from v1:
    - Facts are stored as meaningful PHRASES, not just single keywords.
      This means validator can match 'python developer' or 'pes university'
      instead of requiring a 4-word chunk from a JSON dump.
    - Education entries are flattened to multiple partial strings so
      any reasonable formatting of the same data will match.
    """
    atoms = set()

    def add(s):
        """Normalise and add a string to the atom set."""
        s = str(s).strip().lower()
        if s:
            atoms.add(s)

    # --- Name & contact ---
    add(bank.get('name', ''))
    for v in bank.get('contact', {}).values():
        add(v)

    # --- Skills ---
    # --- Skills ---
    for s in bank.get('skills', []):
        add(s)
        add(expand_abbreviations(s))
    # Also add the abbreviated form so both "ml" and "machine learning" match
        s_lower = s.strip().lower()
        for abbrev, full in ABBREV_MAP.items():
            if s_lower == full:
                add(abbrev)   # store "ml" alongside "machine learning"
            elif s_lower == abbrev:
                add(full)     # store "machine learning" alongside "ml"

    # --- Education ---
    # Store each field individually AND combined so any formatting matches
    for edu in bank.get('education', []):
        add(edu.get('degree', ''))
        add(edu.get('institution', ''))
        add(edu.get('period', ''))
        add(edu.get('score', ''))
        # Combined form that the resume output will likely produce
        combo = f"{edu.get('degree','')} {edu.get('institution','')} {edu.get('period','')}"
        add(combo)

    # --- Experience ---
    for exp in bank.get('experience', []):
        add(exp.get('title', ''))
        add(exp.get('company', ''))
        add(exp.get('period', ''))
        for pt in exp.get('points', []):
            add(pt)

    # --- Projects ---
    for proj in bank.get('projects', []):
        add(proj.get('name', ''))
        add(proj.get('description', ''))
        add(proj.get('summary', ''))
        for t in proj.get('tech', []):
            add(t)
            add(expand_abbreviations(t))

    # --- Achievements & Certifications ---
    for a in bank.get('achievements', []):
        add(a)
    for c in bank.get('certifications', []):
        add(c)

    frozen = frozenset(atoms)
    # SHA-256 seal — proves no facts were added after extraction
    sorted_atoms = sorted(frozen)
    hash_hex = hashlib.sha256('|'.join(sorted_atoms).encode()).hexdigest()

    return frozen, hash_hex


fact_set, fact_hash = build_atomic_fact_set(fact_bank)

print(f'🔒 Atomic fact set built — {len(fact_set)} atoms')
print(f'🔑 Fact bank SHA-256 seal: {fact_hash[:32]}...')
print('\nSample atoms:')
for atom in sorted(list(fact_set))[:20]:
    print(f'  · {atom}')

🔒 Atomic fact set built — 58 atoms
🔑 Fact bank SHA-256 seal: 1293410413e18c034a2d0987d5366b5c...

Sample atoms:
  · 09937889095
  · 2020
  · 2023
  · 90.6%
  · 93.5%
  · apache kafka
  · applied cryptography
  · assembly (basic)
  · b.tech in computer science
  · b.tech in computer science pes university sept 2023 – may 2027
  · bengaluru
  · big data
  · built real-time adaptive streaming pipeline with dynamic topic creation and multi-threaded producer.
  · c
  · cherukuri venkata kartik
  · class x
  · class x mother’s public school 2020
  · class xii
  · class xii mother’s public school 2023
  · cn


## Step 5: Sentence-Level Semantic Matching

**Fix from v1:** Skills were stored as single keywords (e.g. `"Python"`), giving very low cosine similarity (max 0.45) against a full sentence JD.

**Now:** each fact is expanded into a **descriptive sentence** before embedding. This gives much stronger and more meaningful similarity scores.

In [ ]:
# Load the free local embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Embedding model loaded (all-MiniLM-L6-v2)')


# ── Abbreviation expansion ────────────────────────────────────────────────────
ABBREV_MAP = {
    'ml':  'machine learning',
    'dl':  'deep learning',
    'nlp': 'natural language processing',
    'ai':  'artificial intelligence',
    'cv':  'computer vision',
    'llm': 'large language model',
    'rl':  'reinforcement learning',
    'nn':  'neural network',
    'api': 'application programming interface',
    'sql': 'structured query language',
}

def expand_abbreviations(text: str) -> str:
    """Replace known abbreviations with full forms (whole-word, case-insensitive)."""
    for abbrev, full in ABBREV_MAP.items():
        text = re.sub(rf'\b{abbrev}\b', full, text, flags=re.IGNORECASE)
    return text


def build_rich_fact_sentences(bank: dict) -> list[dict]:
    """
    Convert the fact bank into rich descriptive sentences for embedding.
    Abbreviations are expanded before embedding so 'ML' and 'machine learning'
    land in the same semantic space and score equally against a JD.
    """
    sentences = []

    # Skills → descriptive sentences (abbreviations expanded)
    for skill in bank.get('skills', []):
        expanded = expand_abbreviations(skill)
        sentences.append({
            'type':     'skill',
            'fact':     skill,
            'sentence': f'Proficient in {expanded}',
            'source':   'skills'
        })

    # Projects → rich descriptions (abbreviations expanded in desc + tech)
    for proj in bank.get('projects', []):
        name = proj.get('name', '')
        desc = expand_abbreviations(proj.get('description', ''))
        tech = ', '.join(expand_abbreviations(t) for t in proj.get('tech', []))
        sentence = f"Project: {name}. {desc} Technologies used: {tech}."
        sentences.append({
            'type':     'project',
            'fact':     proj.get('summary', name),
            'sentence': sentence,
            'source':   'projects'
        })

    # Experience → role-focused sentences (abbreviations expanded in bullet points)
    for exp in bank.get('experience', []):
        title   = exp.get('title', '')
        company = exp.get('company', '')
        for pt in exp.get('points', []):
            sentences.append({
                'type':     'experience',
                'fact':     pt,
                'sentence': f'{title} at {company}: {expand_abbreviations(pt)}',
                'source':   'experience'
            })

    # Education → context-rich sentences
    for edu in bank.get('education', []):
        sentence = (f"Studied {edu.get('degree','')} at {edu.get('institution','')}"
                    f" ({edu.get('period','')}) with {edu.get('score','')}")
        sentences.append({
            'type':     'education',
            'fact':     sentence,
            'sentence': sentence,
            'source':   'education'
        })

    return sentences


def match_job_description(bank: dict, job_desc: str, top_k: int = 7) -> tuple[list, list]:
    """
    Rank resume facts against the JD using sentence embeddings.
    Both the JD and fact sentences are abbreviation-expanded before comparison,
    so 'ML' in the resume matches 'machine learning' in the JD and vice versa.

    Returns:
        top_matches : list of (fact_dict, score) tuples
        gaps        : list of JD keywords not found in fact bank
    """
    rich_facts = build_rich_fact_sentences(bank)
    if not rich_facts:
        print('⚠️ No facts to match.')
        return [], []

    sentences = [f['sentence'] for f in rich_facts]

    # Expand abbreviations in JD before encoding so embeddings align
    jd_expanded = expand_abbreviations(job_desc)

    jd_emb    = embed_model.encode(jd_expanded, convert_to_tensor=True)
    fact_embs = embed_model.encode(sentences,   convert_to_tensor=True)
    scores    = util.cos_sim(jd_emb, fact_embs)[0].tolist()

    ranked      = sorted(zip(rich_facts, scores), key=lambda x: x[1], reverse=True)
    top_matches = ranked[:top_k]

    # --- Gap Analysis ---
    # Expand abbreviations on both sides before keyword matching so
    # 'ml' in fact_set correctly satisfies 'machine learning' from the JD
    jd_tokens  = re.findall(r'\b[a-zA-Z][a-zA-Z+#.]{2,}\b', jd_expanded.lower())
    # NEW — unigrams + bigrams, but filter bigrams where either word is a stop word
    stop_words = {'for', 'with', 'and', 'the', 'looking', 'developer', 'developer',
                  'experience', 'our', 'you', 'python', 'using', 'that', 'who',
                  'seeking', 'we', 'are', 'have', 'strong', 'good', 'knowledge'}

    jd_tokens   = re.findall(r'\b[a-zA-Z][a-zA-Z+#.]{2,}\b', jd_expanded.lower())
    jd_unigrams = set(jd_tokens) - stop_words
    jd_bigrams  = {
        f'{jd_tokens[i]} {jd_tokens[i+1]}'
        for i in range(len(jd_tokens) - 1)
        if jd_tokens[i] not in stop_words and jd_tokens[i+1] not in stop_words
    }
    jd_words = jd_unigrams | jd_bigrams
    jd_keywords = jd_words - stop_words

    # Build an expanded version of fact_set where every atom also has its
    # abbreviations replaced — this means 'ml' becomes 'machine learning'
    # so a JD keyword 'machine learning' will find it
    expanded_fact_set = fact_set | {expand_abbreviations(a) for a in fact_set}

    gaps = [kw for kw in jd_keywords
            if not any(kw in atom for atom in expanded_fact_set)]

    return top_matches, gaps


job_description = 'Looking for a Python developer with machine learning and data engineering experience'

top_matches, gaps = match_job_description(fact_bank, job_description)

print(f'🎯 Top {len(top_matches)} matching facts (sentence-level embeddings):\n')
for fact_dict, score in top_matches:
    print(f'  [{score:.3f}] [{fact_dict["type"].upper():10s}] {fact_dict["sentence"][:90]}')

print(f'\n📋 Gap analysis — {len(gaps)} JD keywords not in fact bank:')
for g in gaps:
    print(f'  ✗ {g}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded (all-MiniLM-L6-v2)
🎯 Top 7 matching facts (sentence-level embeddings):

  [0.585] [SKILL     ] Proficient in Python
  [0.495] [SKILL     ] Proficient in Machine Learning
  [0.404] [SKILL     ] Proficient in Big Data
  [0.371] [EDUCATION ] Studied B.Tech in Computer Science at PES University (Sept 2023 – May 2027) with GPA: 8.55
  [0.333] [PROJECT   ] Project: FullStack AI Assistant. Developed MERN-based artificial intelligence assistant wi
  [0.325] [SKILL     ] Proficient in Applied Cryptography
  [0.314] [SKILL     ] Proficient in DSA

📋 Gap analysis — 2 JD keywords not in fact bank:
  ✗ engineering
  ✗ data engineering


## Step 6: Build Template-Based Summary (Eliminates Hallucinated Summary)

**Fix from v1:** The v1 Summary was LLM-generated free text — not traceable to any single fact.

**Now:** The Summary is assembled **deterministically** from locked fact bank fields. The LLM cannot add anything to it.

In [ ]:
def build_locked_summary(bank: dict, top_matches: list) -> str:
    """
    Build a professional summary PURELY from locked facts — no LLM generation.

    FIX from v1:
    v1 let the LLM write a free-form Summary, which produced sentences like
    'Python developer with experience in Big Data and AI-related projects'
    that were NOT verbatim in the fact bank, causing validator false positives.

    This function produces the Summary deterministically:
    - Picks the top-ranked skill
    - Picks the top-ranked project names
    - Assembles using a fixed template — zero LLM involvement
    """
    name        = bank.get('name', 'Candidate')
    skills      = bank.get('skills', [])

    # Pick top skills (by order they appear — already ranked by JD in top_matches)
    top_skill_facts = [f for f, _ in top_matches if f['type'] == 'skill']
    top_skills  = [f['fact'] for f in top_skill_facts[:3]]
    if not top_skills:
        top_skills = skills[:3]

    # Pick top project names
    top_proj_facts = [f for f, _ in top_matches if f['type'] == 'project']
    top_proj_names = [f['fact'].split(':')[0].strip() for f in top_proj_facts[:2]]

    # Deterministic template — every word here is directly from the fact bank
    skill_str = ', '.join(top_skills)
    proj_str  = ' and '.join(top_proj_names) if top_proj_names else 'multiple projects'

    edu = bank.get('education', [])
    degree = edu[0].get('degree', '') if edu else ''
    inst   = edu[0].get('institution', '') if edu else ''

    summary = (f"{degree} student at {inst} with hands-on experience in "
               f"{skill_str}, demonstrated through {proj_str}.")
    return summary


locked_summary = build_locked_summary(fact_bank, top_matches)
print('🔒 Locked Summary (assembled from facts, zero LLM):')
print(f'   {locked_summary}')

🔒 Locked Summary (assembled from facts, zero LLM):
   B.Tech in Computer Science student at PES University with hands-on experience in Python, Machine Learning, Big Data, demonstrated through FullStack AI Assistant.


## Step 7: Constraint-Driven Resume Rewriting with Reject-and-Retry Loop

**Two fixes from v1:**
1. The rewrite prompt now receives the **locked summary** — the LLM cannot write a new one.
2. If the validator finds issues, the output is **regenerated** with the flagged claims added as an explicit blocklist — up to `MAX_RETRIES` times.

In [ ]:
def rewrite_resume(bank: dict, job_desc: str, top_matches: list,
                   locked_summary: str, blocklist: list = None) -> str:
    """
    Constrained resume rewrite.

    FIX from v1:
    - locked_summary is injected directly — LLM cannot alter it
    - blocklist is populated on retry with previously flagged hallucinations
    - Temperature stays at 0.1 for all attempts
    """
    top_facts_str = '\n'.join(
        f'- [{f["type"].upper()}] {f["fact"]}' for f, _ in top_matches
    )

    blocklist_str = ''
    if blocklist:
        blocklist_str = (
            '\nBLOCKLIST — these exact phrases appeared in a previous output and were '
            'NOT traceable to the fact bank. They are FORBIDDEN in this output:\n'
            + '\n'.join(f'  FORBIDDEN: "{item}"' for item in blocklist)
        )

    prompt = f"""You are a professional resume writer. Rewrite the resume below.

═══════════════════════════════════════
IMMUTABLE RULES — NO EXCEPTIONS
═══════════════════════════════════════
PERMITTED operations:
  ✓ Rephrase a sentence for professional clarity
  ✓ Reorder sections for better JD alignment
  ✓ Emphasize facts that match the JD
  ✓ Omit irrelevant facts

FORBIDDEN operations:
  ✗ Adding any skill, tool, company, role, or metric NOT in the FACT BANK
  ✗ Inflating achievement percentages or numbers
  ✗ Writing a Summary other than the LOCKED SUMMARY provided below
  ✗ Combining facts to imply experiences that are not stated
{blocklist_str}

═══════════════════════════════════════
LOCKED SUMMARY — copy this verbatim, do not modify:
═══════════════════════════════════════
{locked_summary}

═══════════════════════════════════════
TARGET JOB DESCRIPTION:
═══════════════════════════════════════
{job_desc}

═══════════════════════════════════════
TOP FACTS TO PRIORITIZE (lead with these):
═══════════════════════════════════════
{top_facts_str}

═══════════════════════════════════════
FULL FACT BANK (use ONLY these facts):
═══════════════════════════════════════
{json.dumps(bank, indent=2)}

Output the rewritten resume in plain text. No markdown, no bold, no bullet symbols other than '-'.
"""

    return call_groq(
        prompt,
        system_msg=('You are a resume writer operating under hard factual constraints. '
                    'Every word you write must trace back to the FACT BANK. '
                    'Never invent, never embellish.')
    )


# Initial rewrite (no blocklist yet)
rewritten_resume = rewrite_resume(fact_bank, job_description, top_matches, locked_summary)
print('📄 Initial rewrite produced')
print('─' * 60)
print(rewritten_resume)

📄 Initial rewrite produced
────────────────────────────────────────────────────────────
B.Tech in Computer Science student at PES University with hands-on experience in Python, Machine Learning, Big Data, demonstrated through FullStack AI Assistant.

Contact:
email: karthikcv72@gmail.com
phone: 09937889095
linkedin: LinkedIn
location: Bengaluru

Education:
- Studied B.Tech in Computer Science at PES University from Sept 2023 to May 2027 with GPA: 8.55/10
- Completed Class XII from Mother’s Public School in 2023 with 90.6%
- Completed Class X from Mother’s Public School in 2020 with 93.5%

Skills:
- Python
- Machine Learning
- Big Data
- Applied Cryptography
- DSA

Projects:
- FullStack AI Assistant: Developed MERN-based AI assistant with API integration using MongoDB, ExpressJS, ReactJS, NodeJS
- Dynamic Content Stream with Kafka: Built real-time adaptive streaming pipeline with dynamic topic creation and multi-threaded producer using Python, Apache Kafka, MySQL, Flask
- Real-Time Serv

## Step 8: Field-Level Hallucination Validator

**The core fix from v1.** The v1 validator caused 4 false positives because it:
- Compared full output lines against `json.dumps(fact_bank)` as a string
- Education lines formatted as `'B.Tech, PES University (2023-2027)'` never matched JSON keys like `{"institution": "PES University"}`

**v2 fixes this with three layers:**
1. **Token overlap check** — every content word in an output line must appear somewhere in the atomic fact set
2. **Entity whitelist** — names, institutions, companies from the fact bank are pre-extracted and used for direct lookup
3. **Locked-summary bypass** — the locked summary is never flagged (it was built from facts deterministically)

In [ ]:
def build_entity_whitelist(bank: dict) -> set:
    """
    Extract all named entities (names, institutions, companies, tech names)
    from the fact bank as a flat normalised set of tokens.

    Used for fast token-level lookup during validation.
    """
    entities = set()

    def tokenise(s):
        return set(re.findall(r'\b[a-zA-Z0-9][a-zA-Z0-9+#.]{1,}\b', str(s).lower()))

    entities |= tokenise(bank.get('name', ''))
    for v in bank.get('contact', {}).values():
        entities |= tokenise(v)
    for s in bank.get('skills', []):
        entities |= tokenise(s)
    for edu in bank.get('education', []):
        for v in edu.values():
            entities |= tokenise(v)
    for exp in bank.get('experience', []):
        entities |= tokenise(exp.get('title', ''))
        entities |= tokenise(exp.get('company', ''))
        for pt in exp.get('points', []):
            entities |= tokenise(pt)
    for proj in bank.get('projects', []):
        entities |= tokenise(proj.get('name', ''))
        entities |= tokenise(proj.get('description', ''))
        for t in proj.get('tech', []):
            entities |= tokenise(t)
    for a in bank.get('achievements', []):
        entities |= tokenise(a)

    # Remove generic words that should not count as entities
    stop = {'in','of','at','to','and','for','the','with','using','a','an','is','are',
            'was','be','by','on','from','as','it','its','that','this','or','but',
            'based','class','gpa','percentage','tech','project','experience','developed',
            'built','implemented','multi','real','time','full','stack','pipeline'}
    return entities - stop


def validate_output(bank: dict, rewritten_text: str,
                    locked_summary: str, fact_set: frozenset) -> tuple[list, list]:
    """
    Field-level hallucination validator.

    For each non-trivial output line:
    1. Skip if it's the locked summary (it was built from facts, never hallucinated)
    2. Extract content tokens (nouns, names, tech, numbers)
    3. Check each token against the entity whitelist
    4. Flag lines where >1 content token is NOT in the whitelist

    Returns:
        confirmed_hallucinations : lines with genuinely unknown content
        warnings                 : lines with minor token mismatches (for info only)

    FIX from v1:
    v1 used 4-gram matching against json.dumps() — caused 4 false positives
    because Education formatting differed from JSON representation.
    v2 uses token-level entity matching — 'pes university' matches whether
    the output says 'PES University (Sept 2023)' or just 'PES University'.
    """
    entity_whitelist = build_entity_whitelist(bank)
    stop_words = {'in','of','at','to','and','for','the','with','using','a','an',
                  'is','are','was','be','by','on','from','as','it','this','or',
                  'summary','skills','education','projects','experience','technical',
                  'contact','name','gpa','class','percentage','sept','may',
                  'jan','feb','mar','apr','jun','jul','aug','sep','oct','nov','dec'}

    confirmed_hallucinations = []
    warnings = []

    locked_summary_lower = locked_summary.lower().strip()

    for line in rewritten_text.split('\n'):
        line_clean = line.strip()
        line_lower = line_clean.lower()

        # Skip blank lines, very short lines, section headers
        if not line_clean or len(line_clean) < 8:
            continue
        # Skip if this IS the locked summary — it's fact-derived
        if line_lower in locked_summary_lower or locked_summary_lower in line_lower:
            continue
        # Skip pure bullet headers like 'Technical Skills:'
        if line_clean.endswith(':') and len(line_clean) < 30:
            continue

        # Extract content tokens — numbers, capitalized words, known tech names
        content_tokens = set(re.findall(r'\b[a-zA-Z0-9][a-zA-Z0-9+#.]{1,}\b', line_lower))
        content_tokens -= stop_words

        unknown = [tok for tok in content_tokens if tok not in entity_whitelist]

        if len(unknown) > 2:   # >2 unknown tokens = likely hallucination
            confirmed_hallucinations.append({
                'line': line_clean,
                'unknown_tokens': unknown
            })
        elif len(unknown) == 1 or len(unknown) == 2:
            warnings.append({
                'line': line_clean,
                'unknown_tokens': unknown
            })

    return confirmed_hallucinations, warnings


hallucinations, warnings = validate_output(
    fact_bank, rewritten_resume, locked_summary, fact_set
)

print('\n🔍 VALIDATION RESULTS')
print('═' * 60)
if not hallucinations:
    print('✅ No confirmed hallucinations detected!')
else:
    print(f'❌ {len(hallucinations)} confirmed hallucination(s):')
    for h in hallucinations:
        print(f'  LINE: {h["line"][:80]}')
        print(f'  UNKNOWN TOKENS: {h["unknown_tokens"]}')

if warnings:
    print(f'\n⚠️  {len(warnings)} minor warning(s) (1-2 unknown tokens — likely formatting):')
    for w in warnings:
        print(f'  · {w["line"][:80]}')
        print(f'    unknown: {w["unknown_tokens"]}')
else:
    print('✅ No warnings either — clean output!')


🔍 VALIDATION RESULTS
════════════════════════════════════════════════════════════
❌ 3 confirmed hallucination(s):
  LINE: - Dynamic Content Stream with Kafka: Built real-time adaptive streaming pipeline
  UNKNOWN TOKENS: ['pipeline', 'multi', 'time', 'built', 'real']
  LINE: - Real-Time Server Monitoring Pipeline: Multi-node monitoring of CPU, memory, di
  UNKNOWN TOKENS: ['pipeline', 'multi', 'time', 'real']
  LINE: - Parcel Tracking System: Full-stack delivery management system with real-time t
  UNKNOWN TOKENS: ['full', 'real', 'stack', 'time']

⚠️  7 minor warning(s) (1-2 unknown tokens — likely formatting):
  · email: karthikcv72@gmail.com
    unknown: ['email']
  · phone: 09937889095
    unknown: ['phone']
  · location: Bengaluru
    unknown: ['location']
  · - Studied B.Tech in Computer Science at PES University from Sept 2023 to May 202
    unknown: ['studied']
  · - Completed Class XII from Mother’s Public School in 2023 with 90.6%
    unknown: ['completed']
  · - Completed C

## Step 9: Reject-and-Retry Loop

**NEW in v2.** If hallucinations were found, regenerate with those claims added to the blocklist. Repeat up to `MAX_RETRIES` times.

This makes validation **preventive** (the blocklist prevents re-introduction), not just detective.

In [ ]:
def retry_until_clean(bank, job_desc, top_matches, locked_summary,
                      initial_output, initial_hallucinations, fact_set,
                      max_retries=MAX_RETRIES):
    """
    If the initial rewrite has hallucinations, regenerate with a blocklist.

    FIX from v1:
    v1 returned the output regardless of validation result.
    v2 enters a retry loop: each failed attempt adds the flagged lines
    to an explicit BLOCKLIST in the next prompt, preventing re-introduction.
    """
    current_output        = initial_output
    current_hallucinations = initial_hallucinations
    cumulative_blocklist  = []

    for attempt in range(1, max_retries + 1):
        if not current_hallucinations:
            print(f'✅ Clean output achieved on attempt {attempt - 1 if attempt > 1 else "initial"}')
            break

        print(f'\n🔄 Retry {attempt}/{max_retries} — adding {len(current_hallucinations)} '
              f'flagged claim(s) to blocklist...')

        # Accumulate blocklist across retries
        for h in current_hallucinations:
            cumulative_blocklist.append(h['line'])

        # Regenerate with the blocklist
        current_output = rewrite_resume(
            bank, job_desc, top_matches, locked_summary,
            blocklist=cumulative_blocklist
        )

        # Re-validate
        current_hallucinations, _ = validate_output(
            bank, current_output, locked_summary, fact_set
        )

        print(f'   After retry {attempt}: {len(current_hallucinations)} hallucination(s) remaining')

    else:
        print(f'\n⚠️ After {max_retries} retries, '
              f'{len(current_hallucinations)} issue(s) remain. '
              'Returning best available output.')

    return current_output, current_hallucinations


final_resume, final_hallucinations = retry_until_clean(
    fact_bank, job_description, top_matches, locked_summary,
    rewritten_resume, hallucinations, fact_set
)

print('\n' + '═' * 60)
print('📄 FINAL REWRITTEN RESUME')
print('═' * 60)
print(final_resume)


🔄 Retry 1/3 — adding 3 flagged claim(s) to blocklist...
   After retry 1: 0 hallucination(s) remaining
✅ Clean output achieved on attempt 1

════════════════════════════════════════════════════════════
📄 FINAL REWRITTEN RESUME
════════════════════════════════════════════════════════════
B.Tech in Computer Science student at PES University with hands-on experience in Python, Machine Learning, Big Data, demonstrated through FullStack AI Assistant.

Contact:
email: karthikcv72@gmail.com
phone: 09937889095
linkedin: LinkedIn
location: Bengaluru

Education:
- Studied B.Tech in Computer Science at PES University from Sept 2023 to May 2027 with GPA: 8.55/10
- Completed Class XII from Mother’s Public School in 2023 with 90.6%
- Completed Class X from Mother’s Public School in 2020 with 93.5%

Skills:
- Python
- Machine Learning
- Big Data
- Applied Cryptography
- DSA
- C
- MySQL
- HTML/CSS
- Verilog (Basic)
- Assembly (Basic)
- ReactJS
- NodeJS
- MongoDB
- Neo4j
- postgreSQL
- OS
- CN
- DBMS


## Step 10: Generate Change Log

In [ ]:
def generate_changelog(original_text: str, rewritten_text: str, job_desc: str) -> str:
    """Generate an explainable changelog of all modifications."""
    prompt = f"""Compare these two resume versions and generate a concise CHANGE LOG.

For each change state:
- What was changed
- Why (which rule triggered it)
- How it improves alignment with the job description

Also list any sections that were deprioritized and why.

TARGET JOB: {job_desc}

ORIGINAL RESUME:
{original_text[:2000]}

REWRITTEN RESUME:
{rewritten_text[:2000]}
"""
    return call_groq(prompt)


changelog = generate_changelog(resume_text, final_resume, job_description)
print('📋 CHANGE LOG')
print('─' * 60)
print(changelog)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khpbvpr1ep88pej5kjrbxfth` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98355, Requested 2813. Please try again in 16m49.151999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Step 11: Full Pipeline — All in One

In [ ]:
def run_pipeline(resume_path: str, job_desc: str):
    """
    Full truth-preserving resume rewriter pipeline — v2.

    Steps:
    1  Load resume
    2  Extract structured fact bank (JSON)
    3  Build atomic fact set + SHA-256 seal (hard lock)
    4  Sentence-level semantic matching against JD
    5  Build locked summary (deterministic, no LLM)
    6  Constraint-driven rewrite
    7  Field-level validation
    8  Reject-and-retry loop if hallucinations found
    9  Generate explainable changelog
    """
    SEP = '═' * 60
    print(f'\n{SEP}')
    print('🚀 TRUTH-PRESERVING RESUME REWRITER v2')
    print(SEP)

    print('\n[1/9] Loading resume...')
    text = load_resume(resume_path)
    print(f'      {len(text)} characters loaded')

    print('[2/9] Extracting structured fact bank...')
    bank = extract_fact_bank(text)
    if not bank:
        print('❌ Fact extraction failed. Aborting.')
        return None
    print(f'      {len(bank.get("skills",[]))} skills | '
          f'{len(bank.get("projects",[]))} projects | '
          f'{len(bank.get("education",[]))} education entries')

    print('[3/9] Building atomic fact set + SHA-256 seal...')
    f_set, f_hash = build_atomic_fact_set(bank)
    print(f'      {len(f_set)} atoms | seal: {f_hash[:24]}...')

    print('[4/9] Sentence-level semantic matching...')
    matches, gaps = match_job_description(bank, job_desc)
    print(f'      Top match: {matches[0][0]["sentence"][:60]}... ({matches[0][1]:.3f})')
    if gaps:
        print(f'      Gap analysis: candidate is missing [{", ".join(gaps[:5])}] from JD')
    else:
        print('      Gap analysis: no critical gaps detected')

    print('[5/9] Building locked summary (no LLM)...')
    summary = build_locked_summary(bank, matches)
    print(f'      {summary[:80]}...')

    print('[6/9] Constraint-driven rewrite...')
    rewritten = rewrite_resume(bank, job_desc, matches, summary)
    print('      Rewrite complete')

    print('[7/9] Field-level hallucination validation...')
    h_list, w_list = validate_output(bank, rewritten, summary, f_set)
    print(f'      {len(h_list)} confirmed hallucinations | {len(w_list)} warnings')

    print('[8/9] Reject-and-retry loop...')
    final, remaining = retry_until_clean(
        bank, job_desc, matches, summary, rewritten, h_list, f_set
    )

    print('[9/9] Generating change log...')
    log = generate_changelog(text, final, job_desc)

    # ── Print final results ──────────────────────────────────────────
    print(f'\n{SEP}')
    print('🔑 FACT BANK INTEGRITY SEAL')
    print(SEP)
    print(f'SHA-256: {f_hash}')
    print('(This proves the fact bank was not modified between extraction and rewriting)')

    print(f'\n{SEP}')
    print('🎯 JD ALIGNMENT — TOP MATCHING FACTS')
    print(SEP)
    for f, s in matches:
        print(f'  [{s:.3f}] {f["sentence"][:85]}')

    if gaps:
        print(f'\n📋 GAP ANALYSIS — {len(gaps)} JD skills not found in candidate profile:')
        for g in gaps:
            print(f'  ✗ {g}')

    print(f'\n{SEP}')
    print('📄 FINAL REWRITTEN RESUME')
    print(SEP)
    print(final)

    print(f'\n{SEP}')
    print('🔍 FINAL VALIDATION RESULT')
    print(SEP)
    if not remaining:
        print('✅ ZERO hallucinations detected — output is fully fact-grounded')
    else:
        print(f'⚠️  {len(remaining)} issue(s) after {MAX_RETRIES} retries:')
        for h in remaining:
            print(f'  · {h["line"][:80]}')

    print(f'\n{SEP}')
    print('📋 CHANGE LOG')
    print(SEP)
    print(log)

    return {
        'fact_bank'   : bank,
        'fact_set'    : f_set,
        'fact_hash'   : f_hash,
        'top_matches' : matches,
        'gaps'        : gaps,
        'summary'     : summary,
        'final_resume': final,
        'hallucinations': remaining,
        'changelog'   : log
    }


# ▶️ RUN THE FULL PIPELINE
result = run_pipeline(
    resume_path='/content/Kartik_resume.pdf',
    job_desc='Looking for a Python developer with machine learning and data engineering experience'
)


════════════════════════════════════════════════════════════
🚀 TRUTH-PRESERVING RESUME REWRITER v2
════════════════════════════════════════════════════════════

[1/9] Loading resume...
      1397 characters loaded
[2/9] Extracting structured fact bank...
      18 skills | 4 projects | 3 education entries
[3/9] Building atomic fact set + SHA-256 seal...
      58 atoms | seal: 1293410413e18c034a2d0987...
[4/9] Sentence-level semantic matching...
      Top match: Proficient in Python... (0.585)
      Gap analysis: candidate is missing [data engineering, engineering] from JD
[5/9] Building locked summary (no LLM)...
      B.Tech in Computer Science student at PES University with hands-on experience in...
[6/9] Constraint-driven rewrite...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01khpbvpr1ep88pej5kjrbxfth` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99795, Requested 3400. Please try again in 46m0.48s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}